# Multimodal AI -- Blood Work Analysis

Use a vision-capable LLM to read and interpret it

In [22]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

## Encode the image and send to the vision model

In [16]:
with open("blood_work.png","rb") as f:
    image_b64=base64.b64encode(f.read()).decode()
image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [17]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url":{"url":f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report Extract all test results and flag any values outside the normal range."}
])

response=llm.invoke([message])
print(response.content)


### **Extracted Test Results with Flagged Abnormal Values**  

#### **COMPLETE BLOOD COUNT (CBC)**  
- **Hemoglobin**: **15.1 g/dL** (Normal: 13.5-17.5) **- Normal**  
- **Hematocrit**: **44%** (Normal: 41-53) **- Normal**  
- **WBC**: **6.8 × 10^3/uL** (Normal: 4.5-11.0) **- Normal**  
- **Platelets**: **220 × 10^3/uL** (Normal: 150-400) **- Normal**  

#### **LIPID PANEL**  
- **Total Cholesterol**: **238 mg/dL** (Normal: <200) **- Elevated**  
- **LDL Cholesterol**: **162 mg/dL** (Normal: <100) **- Elevated**  
- **HDL Cholesterol**: **36 mg/dL** (Normal: >40) **- Low**  
- **Triglycerides**: **188 mg/dL** (Normal: <150) **- Elevated**  

#### **METABOLIC PANEL**  
- **Glucose (Fasting)**: **92 mg/dL** (Normal: 70-99) **- Normal**  
- **HbA1c**: **5.3%** (Normal: <5.7) **- Normal**  
- **Creatinine**: **1.0 mg/dL** (Normal: 0.7-1.3) **- Normal**  
- **eGFR**: **82 mL/min** (Normal: >60) **- Normal**  

#### **LIVER FUNCTION**  
No results provided.  

### **Summary of Abnormal Value

# Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool

In [20]:
@ tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
                "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
                "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
           },
            "high_sugar": {
                "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
                "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
            },
            "normal": {
                "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
                "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
           },
        }
    return diet_plans.get(condition, diet_plans["normal"])


In [23]:
SYSTEM_PROMPT="""
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize the condition as one of:
normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan
"""
diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [26]:
result = diet_agent.invoke({
    "messages" : [HumanMessage(content=[
        {"type": "image_url", "image_url":{"url":f"data:image/png;base64,{image_b64}"}},
        {"type": "text", "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})
print(result["messages"][-1].content)

The patient's blood work report indicates high cholesterol levels, with a total cholesterol level of 238 mg/dL and an LDL cholesterol level of 162 mg/dL. Based on this information, I recommend a diet plan for high cholesterol.

The diet plan includes eating foods that are low in saturated and trans fats, cholesterol, and added sugars. The patient should focus on consuming fruits, vegetables, whole grains, and lean protein sources. It is also recommended to limit or avoid foods high in saturated and trans fats, such as red meat, fried foods, full-fat dairy products, and processed snacks.
